In [1]:
!pip install torchmetrics
!pip install --upgrade "sympy<1.13"
!pip install --upgrade transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 19.0 MB/s  0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu130 requires sympy>=1.13.3, but you have sympy 1.12.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 27.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 25.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !cd drive/MyDrive/Master_Thesis/datasets/ && unzip train.zip
# !cd drive/MyDrive/Master_Thesis/datasets/gtFine/ && unzip train.zip

Streaming output truncated to the last 5000 lines.
  inflating: train/krefeld/krefeld_000000_003707_gtFine_polygons.json  
  inflating: train/krefeld/krefeld_000000_003937_gtFine_color.png  
  inflating: train/krefeld/krefeld_000000_003937_gtFine_instanceIds.png  
  inflating: train/krefeld/krefeld_000000_003937_gtFine_labelIds.png  
  inflating: train/krefeld/krefeld_000000_003937_gtFine_polygons.json  
  inflating: train/krefeld/krefeld_000000_004447_gtFine_color.png  
  inflating: train/krefeld/krefeld_000000_004447_gtFine_instanceIds.png  
  inflating: train/krefeld/krefeld_000000_004447_gtFine_labelIds.png  
  inflating: train/krefeld/krefeld_000000_004447_gtFine_polygons.json  
  inflating: train/krefeld/krefeld_000000_004608_gtFine_color.png  
  inflating: train/krefeld/krefeld_000000_004608_gtFine_instanceIds.png  
  inflating: train/krefeld/krefeld_000000_004608_gtFine_labelIds.png  
  inflating: train/krefeld/krefeld_000000_004608_gtFine_polygons.json  
  inflating: train/kre

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import AutoModel

# ==========================================
# 1. Cityscapes PyTorch Dataset Wrapper
# ==========================================
class CityscapesFeatureDataset(Dataset):
    """
    Dataset loader for Cityscapes leftImg8bit images.
    Expects directory structure: root_dir/leftImg8bit/train (or val)
    """
    def __init__(self, root_dir: str, split: str = "train", img_size: tuple = (1024, 2048)):
        # UPDATED LINE: Added "leftImg8bit" folder to the path builder to align with Cityscapes format
        self.split_dir = os.path.join(root_dir, split)
        self.img_size = img_size
        self.image_paths = []

        if not os.path.exists(self.split_dir):
            raise FileNotFoundError(f"Cityscapes split directory not found: {self.split_dir}")

        # populating the image_paths
        for city in os.listdir(self.split_dir):
            city_path = os.path.join(self.split_dir, city)
            if os.path.isdir(city_path):
                for file_name in os.listdir(city_path):
                    if file_name.endswith("_leftImg8bit.png"):
                        self.image_paths.append(os.path.join(city_path, file_name))

        print(f"Found {len(self.image_paths)} images in Cityscapes '{split}' split.")

        self.transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        tensor_img = self.transform(image)

        rel_path = os.path.relpath(img_path, self.split_dir)
        return tensor_img, rel_path


# ==========================================
# 2. DINOv3 Feature Extractor Pipeline
# ==========================================
def extract_and_save_dinov3_features(
    cityscapes_root: str,
    output_dir: str,
    split: str = "train",
    model_name: str = "dinov3_vitb16",
    batch_size: int = 8,
    img_size: tuple = (1024, 2048),
    hf_token: str = None  # UPDATED LINE: Added explicit hf_token to function signature
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load DINOv3 Backbone via HuggingFace
    print(f"Loading DINOv3 backbone: {model_name}...")
    model = AutoModel.from_pretrained(
        "facebook/dinov3-vitb16-pretrain-lvd1689m",
        token=hf_token  # UPDATED LINE: Uses the passed-down function argument safely
    )
    model = model.to(device)
    model.eval()

    for param in model.parameters():
        param.requires_grad = False

    dataset = CityscapesFeatureDataset(root_dir=cityscapes_root, split=split, img_size=img_size)
    # UPDATED LINE: Added pin_memory=True to accelerate GPU tensor transfer
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    patch_size = 16
    h_patches = img_size[0] // patch_size
    w_patches = img_size[1] // patch_size

    print(f"Extracting DINOv3 spatial feature grid shape: ({h_patches}, {w_patches}, D)...")

    with torch.no_grad():
        for imgs, rel_paths in tqdm(dataloader, desc=f"Extracting DINOv3 features for {split}"):
            imgs = imgs.to(device)

            # UPDATED LINE: Rename output to match Hugging Face naming conventions
            outputs = model(imgs)

            # UPDATED LINE: Replaced string dictionary lookup with standard .last_hidden_state object access.
            num_prefix_tokens = 1 + model.config.num_register_tokens  # 1 CLS + 4 registers = 5 tokens
            patch_tokens = outputs.last_hidden_state[:, num_prefix_tokens:]
            B, N, D = patch_tokens.shape

            assert N == h_patches * w_patches, f"Expected {h_patches*w_patches} patch tokens, got {N}."

            # Reshape into 2D Spatial Grid: (B, h_patches, w_patches, D)
            spatial_features = patch_tokens.reshape(B, h_patches, w_patches, D)

            # Save individual tensor files
            for i in range(B):
                out_file_path = os.path.join(output_dir, split, rel_paths[i]).replace(".png", ".pt")
                os.makedirs(os.path.dirname(out_file_path), exist_ok=True)

                # Save feature tensor (h, w, D) to disk
                torch.save(spatial_features[i].cpu(), out_file_path)

    print(f"\nDINOv3 feature extraction complete! Saved to: {os.path.join(output_dir, split)}")


if __name__ == "__main__":
    # UPDATED LINE: Swapped path to absolute '/content/' path avoiding hidden relative path errors in Colab
    CITYSCAPES_PATH = "/content/drive/MyDrive/Master_Thesis/datasets/"
    OUTPUT_FEATURES_PATH = "/content/drive/MyDrive/Master_Thesis/datasets/cityscapes_dinov3_features"

    extract_and_save_dinov3_features(
        cityscapes_root=CITYSCAPES_PATH,
        output_dir=OUTPUT_FEATURES_PATH,
        split="train",
        model_name="dinov3_vitb16",
        batch_size=8,
        img_size=(1024, 2048),
        hf_token=hf_token
    )


Using device: cuda
Loading DINOv3 backbone: dinov3_vitb16...


config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Found 2762 images in Cityscapes 'train' split.
Extracting DINOv3 spatial feature grid shape: (64, 128, D)...


Extracting DINOv3 features for train: 100%|██████████| 346/346 [1:09:10<00:00, 11.99s/it]


DINOv3 feature extraction complete! Saved to: /content/drive/MyDrive/Master_Thesis/datasets/cityscapes_dinov3_features/train
